# 🔗 02 — Fusion finale (`df_model`)

Découvre automatiquement toutes les tables `dim_*.parquet` dans
`data/processed/` (peu importe qui les a produites ni dans quel ordre) et
les fusionne à `fact_urgences` (la table Y de `00_config_commun.ipynb`).

Prérequis : `00_config_commun.ipynb` + au moins un des `01x_pipeline_*.ipynb`
(chacun dépose son `dim_*.parquet` indépendamment des autres).

La jointure (voir `build_model_view()` plus bas) se décide selon les
colonnes de chaque table : `dept` + `annee_mois` → jointure sur les deux,
`annee_mois` seul (ex : `dim_temps`) → jointure sur `annee_mois`, `dept` seul
(ex : `dim_geo_pop`, `dim_csp`) → jointure sur `dept`. Pas besoin de toucher
ce notebook quand quelqu'un ajoute une table, il suffit que le fichier
`dim_xxx.parquet` existe.

In [10]:
# Préambule : on se place dans le répertoire racine du projet et on ajoute le répertoire courant au PYTHONPATH pour pouvoir importer src/config.py

# pour recharger automatiquement les modules modifiés （src config surtout） sans redémarrer le kernel
%load_ext autoreload 
%autoreload 2

import os
import sys
from pathlib import Path
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
from src.config import RAW_DIR, TABLES_DIR, ANNEE_DEBUT, ANNEE_FIN, DEPTS,  DEPT_NOM_TO_CODE
from src.validation import valider_dim_table

print(
    f"Config chargée depuis src/config.py : {len(DEPTS)} départements | {ANNEE_DEBUT}–{ANNEE_FIN}")
print(f"RAW_DIR    = {RAW_DIR}")
print(f"TABLES_DIR = {TABLES_DIR}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Config chargée depuis src/config.py : 96 départements | 2020–2025
RAW_DIR    = /Users/siranh/Documents/Data Scientest/projet_liora/data/raw
TABLES_DIR = /Users/siranh/Documents/Data Scientest/projet_liora/data/processed


---
## 1. Audit des tables disponibles

In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# AUDIT DES TABLES — découverte automatique de fact_urgences + tous les dim_*
# Même fonction de validation que dans les 01x_pipeline_*.ipynb (src/validation.py),
# pour ne pas avoir deux logiques de contrôle différentes qui divergent.
# ══════════════════════════════════════════════════════════════════════════════
fichiers_a_auditer = [TABLES_DIR / "fact_urgences.parquet"] + sorted(TABLES_DIR.glob("dim_*.parquet"))

print("=" * 60)
print("AUDIT DES TABLES")
print("=" * 60)

for fpath in fichiers_a_auditer:
    nom = fpath.stem
    if not fpath.exists():
        print(f"\n❌ {nom} — fichier absent")
        continue
    df = pd.read_parquet(fpath)
    print(f"\n📋 {nom} : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
    valider_dim_table(df, nom)

print("=" * 60)

AUDIT DES TABLES

📋 fact_urgences : 6,912 lignes × 11 colonnes
── Validation de fact_urgences ──
  ✅ Tous les codes dept sont valides (96 départements)
  ✅ Aucun doublon sur la clé ['dept', 'annee_mois']
  Taux de valeurs manquantes :
    taux_hosp_allergie               0.1%
    taux_sos_allergie               53.5%
    taux_hosp_asthme                 0.1%
    taux_sos_asthme                 53.5%
    taux_hosp_bronchiolite           3.4%
    taux_sos_bronchiolite           53.6%
  ✅ OK — prêt pour la fusion (fact_urgences)


📋 dim_csp : 96 lignes × 8 colonnes
── Validation de dim_csp ──
  ✅ Tous les codes dept sont valides (96 départements)
  ✅ Aucun doublon sur la clé ['dept']
  ✅ OK — prêt pour la fusion (dim_csp)


📋 dim_geo_pop : 96 lignes × 8 colonnes
── Validation de dim_geo_pop ──
  ✅ Tous les codes dept sont valides (96 départements)
  ✅ Aucun doublon sur la clé ['dept']
  ✅ OK — prêt pour la fusion (dim_geo_pop)


📋 dim_indicateurs_contexte : 576 lignes × 10 colonnes
── Val

---
## 2. Fusion en `df_model`

In [9]:
def build_model_view() -> pd.DataFrame:
    """
    Fusionner fact_urgences avec TOUTES les tables dim_*.parquet trouvées dans
    data/processed/.

    Détection de la clé de jointure par table (selon les colonnes présentes) :
        dept + annee_mois → jointure sur les deux (tables mensuelles par dept)
        dept + annee      → jointure sur les deux (tables annuelles par dept)
        annee_mois seul    → jointure sur annee_mois (ex : dim_temps)
        dept seul          → jointure sur dept (ex : dim_geo_pop, dim_csp)
    """

    df = pd.read_parquet(TABLES_DIR / "fact_urgences.parquet")
    print(f"Base : fact_urgences ({df.shape[0]:,} lignes × {df.shape[1]} colonnes)")

    # dim_temps en premier (fournit "annee"), puis le reste par ordre alphabétique
    all_dims = sorted(TABLES_DIR.glob("dim_*.parquet"))
    dim_files = [p for p in all_dims if p.stem == "dim_temps"] + [p for p in all_dims if p.stem != "dim_temps"] #stem pour nom du fichier sans extension

    for fpath in dim_files:
        nom = fpath.stem
        dim = pd.read_parquet(fpath)

        if "dept" in dim.columns and "annee_mois" in dim.columns:
            cle = ["dept", "annee_mois"]
        elif "dept" in dim.columns and "annee" in dim.columns:
            if "annee" not in df.columns:
                print(f" {nom} ignorée : colonne 'annee' pas encore disponible dans df "
                      f"(dim_temps doit être fusionnée avant)")
                continue
            cle = ["dept", "annee"]
        elif "annee_mois" in dim.columns:
            cle = ["annee_mois"]
        elif "dept" in dim.columns:
            cle = ["dept"]
        else:
            print(f" {nom} ignorée : ni 'dept' ni 'annee_mois'/'annee' dans ses colonnes")
            continue

        avant_lignes, avant_cols = df.shape
        df = df.merge(dim, on=cle, how="left")
        print(f"  ✅ {nom:<20} joint sur {cle}  "
              f"(+{df.shape[1] - avant_cols} colonnes, {df.shape[0]:,} lignes)")

    # Taux de remplissage global (les 15 colonnes les plus incomplètes)
    print("\nColonnes les plus incomplètes :")
    missing = (df.isnull().mean() * 100).sort_values(ascending=False)
    for col, pct in missing[missing > 0].head(15).items():
        print(f"  {col:<35} {pct:>5.1f}%")

    print(f"\n df_model : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
    return df


# ══════════════════════════════════════════════════════════════════════════════
# EXÉCUTION
# ══════════════════════════════════════════════════════════════════════════════
df_model = build_model_view()

if not df_model.empty:
    df_model.to_parquet(TABLES_DIR / "df_model.parquet", index=False)
    df_model.to_csv(TABLES_DIR / "df_model.csv", index=False)
    print(f"\nSauvegardée : {TABLES_DIR / 'df_model.parquet'}")
    print("\nColonnes disponibles pour la modélisation :")
    print(list(df_model.columns))
    display(df_model.head(5))

Base : fact_urgences (6,912 lignes × 11 colonnes)
  ✅ dim_temps            joint sur ['annee_mois']  (+12 colonnes, 6,912 lignes)
  ✅ dim_csp              joint sur ['dept']  (+7 colonnes, 6,912 lignes)
  ✅ dim_geo_pop          joint sur ['dept']  (+7 colonnes, 6,912 lignes)
  ✅ dim_indicateurs_contexte joint sur ['dept', 'annee']  (+8 colonnes, 6,912 lignes)
  ✅ dim_meteo            joint sur ['dept', 'annee_mois']  (+6 colonnes, 6,912 lignes)
  ✅ dim_pollen           joint sur ['dept', 'annee_mois']  (+10 colonnes, 6,912 lignes)
  ✅ dim_qualite_air      joint sur ['dept', 'annee_mois']  (+6 colonnes, 6,912 lignes)

Colonnes les plus incomplètes :
  moisissure_alternaria_moy            90.8%
  moisissure_cladosporium_moy          90.8%
  cont_part_csp_cadres                 83.3%
  cont_part_pop_pole_urbain            83.3%
  cont_tx_act                          83.3%
  pollen_artemisia_moy                 81.8%
  pollen_platane_moy                   81.6%
  pollen_betula_moy         

,dept,annee_mois,taux_urgences_allergie,taux_hosp_allergie,taux_sos_allergie,taux_urgences_asthme,taux_hosp_asthme,taux_sos_asthme,taux_urgences_bronchiolite,taux_hosp_bronchiolite,...,pollen_urticacees_moy,moisissure_alternaria_moy,moisissure_cladosporium_moy,pollen_global_max,no_moy,no2_moy,o3_moy,pm10_moy,pm25_moy,so2_moy
0,01,2020-01,815.39,245.12,NaN,402.03,878.26,NaN,23287.66,44861.11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,01,2020-02,722.06,228.66,NaN,462.67,437.93,NaN,10169.56,25875.35,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,01,2020-03,377.48,193.97,NaN,805.16,841.07,NaN,9939.19,14230.77,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,01,2020-04,500.03,98.04,NaN,574.89,863.81,NaN,0.00,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,01,2020-05,568.77,311.94,NaN,522.41,552.69,NaN,0.00,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
